```{contents}
```
## Adadelta Optimizer in Deep Learning

### Overview

**Adadelta** is an adaptive learning rate optimization algorithm designed to overcome the major limitation of **Adagrad**—its aggressively decreasing learning rate.
Adadelta dynamically adjusts learning rates **per parameter**, based on a window of recent gradients, eliminating the need to manually choose a global learning rate.

---

### Motivation

Adagrad accumulates squared gradients:

$$
G_t = \sum_{i=1}^t g_i^2
$$

This causes the effective learning rate:

$$
\eta_t = \frac{\eta}{\sqrt{G_t + \epsilon}}
$$

to shrink continuously → learning stalls.

**Adadelta fixes this** by keeping only an **exponentially decaying average** of past squared gradients.

---

### Core Mechanism

Adadelta maintains two running averages:

| Symbol                   | Meaning                                       |
| ------------------------ | --------------------------------------------- |
| $E[g^2]_t$             | Decaying average of squared gradients         |
| $E[\Delta \theta^2]_t$ | Decaying average of squared parameter updates |

Update rules:

$$
E[g^2]*t = \rho E[g^2]*{t-1} + (1 - \rho) g_t^2
$$

$$
\Delta \theta_t = - \frac{\sqrt{E[\Delta \theta^2]_{t-1} + \epsilon}}
{\sqrt{E[g^2]_t + \epsilon}} g_t
$$

$$
E[\Delta \theta^2]*t = \rho E[\Delta \theta^2]*{t-1} + (1 - \rho)(\Delta \theta_t)^2
$$

---

### Intuition

Adadelta adapts each parameter’s step size based on:

* How large recent gradients have been
* How large previous updates were

This creates **unit-consistent updates** and prevents learning rates from shrinking to zero.

---

### Training Workflow

| Step              | Description                  |
| ----------------- | ---------------------------- |
| Forward           | Compute predictions          |
| Backward          | Compute gradients            |
| Statistics update | Update moving averages       |
| Adaptive step     | Compute per-parameter update |
| Parameter update  | Apply update                 |

---

### PyTorch Demonstration

```python
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(2, 64),
    nn.ReLU(),
    nn.Linear(64, 2)
)

optimizer = optim.Adadelta(model.parameters(), rho=0.9, eps=1e-6)
loss_fn = nn.CrossEntropyLoss()

x = torch.randn(128, 2)
y = torch.randint(0, 2, (128,))

for _ in range(100):
    optimizer.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    optimizer.step()
```

---

### Hyperparameters

| Parameter | Role                                         |
| --------- | -------------------------------------------- |
| `rho`     | Decay rate for moving averages (default 0.9) |
| `eps`     | Numerical stability constant                 |
| `lr`      | Usually left at default (1.0)                |

---

### Variants and Relations

| Optimizer | Key Difference                      |
| --------- | ----------------------------------- |
| Adagrad   | Accumulates all gradients           |
| RMSProp   | Keeps only squared gradient average |
| Adam      | Adds momentum on gradients          |
| Adadelta  | RMSProp + update history            |

---

### When to Use Adadelta

| Scenario                   | Reason                      |
| -------------------------- | --------------------------- |
| No tuning of learning rate | Learning rate-free          |
| Sparse features            | Handles sparse updates well |
| Noisy gradients            | Smooth adaptive steps       |

---

### Summary

| Property              | Adadelta |
| --------------------- | -------- |
| Adaptive learning     | Yes      |
| Per-parameter scaling | Yes      |
| Manual LR needed      | No       |
| Memory cost           | Moderate |
| Convergence stability | High     |

---

### Key Insight

> **Adadelta automatically balances gradient magnitudes and update magnitudes, enabling stable learning without explicit learning rate tuning.**
